# Data retrieval from MUL
This code retrieves list of unit modifications from http://www.masterunitlist.info/Unit/ and their Alpha Strike parameters.

In [3]:
# @title
loaded = False

try:
  loaded = loaded_save
except:
  loaded = False

if not loaded:
  print("LOADING ANCIENT SCRIPTS")
  !python -m pip install tqdm
  !python -m pip install pandas
  !python -m pip install openpyxl
  !python -m pip install jupyter-ui-poll
  import time
  import requests
  from bs4 import BeautifulSoup
  import csv
  from datetime import datetime
  import pandas as pd
  from tqdm.auto import tqdm
  import openpyxl
  from openpyxl.utils import get_column_letter
  from google.colab import output
  import ipywidgets as widgets
  from IPython.display import display, clear_output
  from jupyter_ui_poll import ui_events
  output.clear()
  print("ANCIENT SCRIPTS READY")
loaded_save = True

In [29]:
# @title
try:
  search_val = saved_search
  chassis_val = saved_chassis
except:
  search_val = ""
  chassis_val = ""

search_widget = widgets.Text(value="Rifleman IIC 8", description="Unit Name or Model for search:", style={'description_width': 'initial'}, layout={'width': '30%'})
chassis_widget = widgets.Text(value="RFL-IIC", description="Chassis name, will be used in sheet:", style={'description_width': 'initial'}, layout={'width': '30%'})

cb_bm = widgets.Checkbox(value=True, description='BattleMech')
cb_cv = widgets.Checkbox(value=False, description='Combat Vehicle')
cb_as = widgets.Checkbox(value=False, description='Aerospace')
cb_in = widgets.Checkbox(value=False, description='Infantry')
cb_im = widgets.Checkbox(value=False, description='IndustrialMech')
cb_pm = widgets.Checkbox(value=False, description='Protomech')
cb_sv = widgets.Checkbox(value=False, description='Support Vehicle')
cb_aa = widgets.Checkbox(value=False, description='Advanced Aero')
cb_asup = widgets.Checkbox(value=False, description='Advanced Support')
cb_bld = widgets.Checkbox(value=False, description='Building')
cb_unk = widgets.Checkbox(value=False, description='Unknown')

grid = widgets.GridspecLayout(4, 3)
grid[0,0], grid[0,1], grid[0,2] = cb_bm, cb_cv, cb_as
grid[1,0], grid[1,1], grid[1,2] = cb_in, cb_im, cb_pm
grid[2,0], grid[2,1], grid[2,2] = cb_sv, cb_aa, cb_asup
grid[3,0], grid[3,1] = cb_bld, cb_unk

type_map = {
    cb_bm: "18", cb_cv: "19", cb_as: "17", cb_in: "21",
    cb_im: "20", cb_pm: "23", cb_sv: "24", cb_aa: "81",
    cb_asup: "79", cb_bld: "97", cb_unk: "76"
}

button = widgets.Button(description="SAVE AND PROCEED", button_style='danger', layout={'width': '100%'})
output = widgets.Output()

print("Unit Search, equal to MUL http://www.masterunitlist.info/Unit/")
display(search_widget, chassis_widget)
print("\nUnit types:")
display(grid)
display(button, output)

clicked = False
def on_button_clicked(b):
    global clicked
    clicked = True

button.on_click(on_button_clicked)

with ui_events() as poll:
    while not clicked:
        poll(10)
        time.sleep(0.1)

search_val = saved_search = search_widget.value
chassis_val = saved_chassis = chassis_widget.value
selected_types = "".join([f"&Types={tid}" for cb, tid in type_map.items() if cb.value])

search_url = "http://www.masterunitlist.info/Unit/Filter?Name=" + search_val.replace(" ", "+") + "&MinPV=1&MaxPV=100" + selected_types

print("\nSearch: " + search_url)


Unit Search, equal to MUL http://www.masterunitlist.info/Unit/


Text(value='Rifleman IIC 8', description='Unit Name or Model for search:', layout=Layout(width='30%'), style=D…

Text(value='RFL-IIC', description='Chassis name, will be used in sheet:', layout=Layout(width='30%'), style=De…


Unit types:


GridspecLayout(children=(Checkbox(value=True, description='BattleMech', layout=Layout(grid_area='widget001')),…

Button(button_style='danger', description='SAVE AND PROCEED', layout=Layout(width='100%'), style=ButtonStyle()…

Output()

KeyboardInterrupt: 

Connect and get confirmation of connection

In [6]:
response = requests.get(search_url)
if response.status_code == 200:
    soup = BeautifulSoup(response.text, 'html.parser')
    print("Access granted!")
else:
    print("Failed to retrieve the webpage.")

Access granted!


Make a list of all mechs

In [ ]:
unit_names = [
    a.text for a in soup.find_all('a', href=True)
    if a['href'].startswith("/Unit/Details/")
]

unit_urls = [
    "https://masterunitlist.info" + a['href']
    for a in soup.find_all('a', href=True)
    if a['href'].startswith("/Unit/Details/")
]

paired = list(zip(unit_names, unit_urls))

paired.sort(key=lambda x: int(''.join(filter(str.isdigit, x[0]))) if ''.join(filter(str.isdigit, x[0])) else 0)

unit_names = [name for name, url in paired]
unit_urls  = [url  for name, url in paired]

print(unit_names)

Make a dataframe to collect units parameters

In [ ]:
dataset = pd.DataFrame(columns = ["Name", "Model", "Type", "Role", "Era", "Tech", "Specials", "PV", "Size", "TMM", "Move", "Jump", "Short", "Medium", "Long", "Overheat", "Armor", "Structure", "Tonnage"])

Fill `dataset` with units' information:

In [ ]:
Jump = 0
TMM = 0
MoveI = 0
EraI = 0
Era = "STAR LEAGUE"

for i, unit_name in enumerate(unit_names):
    if (i%50 == 0):
        percent_complete = (i + 1) / len(unit_names) * 100

    # Define the URL with the query parameters
    url = f"https://masterunitlist.azurewebsites.net/Unit/QuickList?Name={unit_name}"

    # Send an HTTP GET request with the parameters
    response = requests.get(url, stream=True)

    # Check if the request was successful (status code 200)
    if response.status_code == 200:

        # Parse the JSON response
        data = response.json()

        # Identify which unit should be parsed as one request may return several units. For example:
        # https://masterunitlist.azurewebsites.net/Unit/QuickList?Name=Phoenix%20Hawk%20PXH-1k

        for item in data.get("Units"):
            if (item.get("Name")==unit_name):
                index = data.get("Units").index(item)

                Move = data.get("Units")[index].get("BFMove", "")
                Move = Move.replace('"',"")
                if "j" in Move:
                    Move = Move.replace('j',"")
                    if "/" in Move:
                        MoveS = Move.split("/")
                        Jump = MoveS[1]
                        Move = MoveS[0]
                    else:
                        Jump = Move
                else:
                    MoveI = Move

        Move = numbers = "".join([char for char in Move if char.isdigit()])
        MoveI = int(Move)
        match MoveI:
            case n if 4 <= n <= 8:
                TMM = 1
            case n if 9 <= n <= 12:
                TMM = 2
            case n if 13 <= n <= 18:
                TMM = 3
            case n if 19 <= n <= 34:
                TMM = 4
            case n if 35 <= n <= 1000:
                TMM = 5
            case _:
                TMM = "?"

        EraI = int(data.get("Units")[index].get("EraId"))
        match EraI:
            case n if n == 9 or n == 10:
                Era = "STAR LEAGUE"
            case n if n == 11 or n == 255 or n == 256:
                Era = "SUCCESSION WARS"
            case n if n == 13:
                Era = "CLAN INVASION"
            case n if n == 247:
                Era = "CIVIL WAR"
            case n if n == 14:
                Era = "JIHAD"
            case n if n == 15 or n == 254 or n == 16:
                Era = "DARK AGE"
            case n if n == 257:
                Era = "ILCLAN"
            case _:
                Era = data.get("Units")[index].get("EraId")

        Tech = data.get("Units")[index].get("Technology").get("Name")
        match Tech:
          case n if n == "Clan":
              Tech = "CLAN"
          case n if n == "Inner Sphere":
              Tech = "IS"
          case n if n == "Primitive":
              Tech = "PRIMITIVE"
          case _:
              Tech = "OTHER"

        Size = data.get("Units")[index].get("BFSize", 0)
        match Size:
          case n if n == 1:
              Size = "1L"
          case n if n == 2:
              Size = "2M"
          case n if n == 3:
              Size = "3H"
          case n if n == 4:
              Size = "4A"
          case _:
              Size = "OTHER"

        # Extract the desired information
        parsed_unit = {
            "Name": ШАССИ,
            "Model": data.get("Units")[index].get("Name"),
            "Type": data.get("Units")[index].get("Type").get("Name").upper(),
            "Role": data.get("Units")[index].get("Role").get("Name").upper(),
            "Era": Era,
            "Tech": Tech,
            "Specials": data.get("Units")[index].get("BFAbilities", ""),
            "PV": data.get("Units")[index].get("BFPointValue", 0),
            "Size": Size,
            "TMM": TMM,
            "Move": Move,
            "Jump": Jump,
            "Short": data.get("Units")[index].get("BFDamageShort", 0),
            "Medium": data.get("Units")[index].get("BFDamageMedium", 0),
            "Long": data.get("Units")[index].get("BFDamageLong", 0),
            "Overheat": data.get("Units")[index].get("BFOverheat", 0),
            "Armor": data.get("Units")[index].get("BFArmor", 0),
            "Structure": data.get("Units")[index].get("BFStructure", 0),
            "Tonnage": data.get("Units")[index].get("FormatedTonnage")
        }

        print(parsed_unit)

        # Add unit parameters to the dataframe
        dataset = pd.concat([dataset, pd.DataFrame([parsed_unit])], ignore_index=True)

print(f"Data has been saved to 'datase' DataFrame")

Clear the dataset and make a new one (to have an access to initial data).
All the units with 0 PV can be dropped as they shouldn't be used during a game

In [ ]:
unitlist = dataset[dataset['PV'] != 0].reset_index(drop=True)
#unitlist.info()

Check all NA values and replace them if applicable

In [ ]:
unitlist[unitlist["Model"].isna()].head()

Specials can be None

In [ ]:
unitlist[unitlist["Specials"].isna()].head()

In [ ]:
path = f"unit_list_{ШАССИ}.xlsx"

unitlist.to_excel(
    path,
    index=False,
    engine='openpyxl'
)

wb = openpyxl.load_workbook(path)
ws = wb.active

from openpyxl.styles import Font, Alignment

data_font = Font(name="Roboto", size=10, color="434343")
hyperlink_font = Font(name="Roboto", size=10, color="1155cc", underline="single")
data_alignment = Alignment(horizontal="left", vertical="top", wrap_text=True)

for row in ws.iter_rows(min_row=1, max_row=ws.max_row, min_col=1, max_col=ws.max_column):
    for cell in row:
        cell.font = data_font
        cell.alignment = data_alignment

model_col = None
for col in range(1, ws.max_column + 1):
    if ws.cell(row=1, column=col).value == "Model":
        model_col = col
        break

if model_col is not None:
    for row_idx in range(2, ws.max_row + 1):
        cell = ws.cell(row=row_idx, column=model_col)

        idx = row_idx - 2
        if idx < len(unit_urls):
            full_url = unit_urls[idx]
        else:
            full_url = None

        if full_url:
            cell.hyperlink = full_url
            cell.font = hyperlink_font

wb.save(path)